In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_INPUTS = True
REUSE_QC = False
REUSE_FIGURES = False
REUSE_REPORT = False


# OpenPlaque — Master Coronary Anatomy + QC v1.1

Drive-layout-safe master report. Scientific anatomy decisions and QC thresholds are unchanged; v1.1 adds robust discovery of the established RCA source centerline when the older source-volume package is nested elsewhere under `MyDrive/OpenPlaque`.


In [ ]:
!pip -q install scipy pandas matplotlib


In [ ]:
import os, shutil, sys
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone -q --depth 1 --branch master-coronary-anatomy-qc-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0, '/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.master_coronary_anatomy_qc_v2 import (
    MasterCoronaryAnatomyQCWorkflow, synthetic_master_qc_self_test
)
test=synthetic_master_qc_self_test(); display(test); assert test['passed'], test
wf=MasterCoronaryAnatomyQCWorkflow(root='/content/drive/MyDrive/OpenPlaque', reuse={
    'inputs':REUSE_INPUTS,'qc':REUSE_QC,'figures':REUSE_FIGURES,'report':REUSE_REPORT
})
display(wf.cache_status())


In [ ]:
frozen=wf.load_frozen_anatomy(); display(frozen); display(wf.artery_summary)
print('Resolved source files:')
for k,v in wf.source_files.items(): print(f'  {k}: {v}')


In [ ]:
qc=wf.build_qc(candidate_step_mm=1.0,max_selected_per_vessel=12)
print('Selected QC planes:',len(qc)); display(qc)


In [ ]:
history=wf.build_history(); display(history)


In [ ]:
names=wf.make_figures()
print('Generated figures:')
for name in names:
    print(name); display(Image(filename=str(wf.out/name)))


In [ ]:
report=wf.build_report(); zip_path=wf.package(); display(wf.artery_summary)
print('HTML report:',report)
print('Final ZIP:',zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_MASTER_CORONARY_ANATOMY_QC_REPORT_BACK.zip')
